# Exercício 04 — Backpropagation do Zero
### Mastering Machine Learning Advanced · ML Engineer Track
### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

## Por que isso importa para um ML Engineer?

Backpropagation é o algoritmo que treina toda rede neural — de MLPs simples até GPT. Frameworks como PyTorch e TensorFlow fazem isso automaticamente com **autograd**, mas entender o que acontece por baixo é fundamental para:

- Implementar loss functions e camadas customizadas
- Diagnosticar problemas de gradiente (vanishing/exploding)
- Entender por que certas arquiteturas funcionam e outras não
- Ler e entender papers de ML

Neste exercício você vai implementar uma rede neural de 2 camadas do zero usando apenas NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
np.random.seed(42)
%matplotlib inline

# dataset: problema não-linear (moons)
X_raw, y_raw = make_moons(n_samples=500, noise=0.2, random_state=42)
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# reshape para coluna
y_tr = y_tr.reshape(-1, 1)
y_te = y_te.reshape(-1, 1)

plt.scatter(X_raw[:, 0], X_raw[:, 1], c=y_raw, cmap='RdBu', alpha=0.6, s=20)
plt.title('Dataset: Make Moons (problema não-linear)')
plt.tight_layout()
plt.show()

---
## Exercício 4.1 — Funções de Ativação

Implemente as funções de ativação e suas derivadas (necessárias para o backprop).

In [ ]:
def sigmoid(z):
    # SEU CÓDIGO AQUI
    # sigmoid(z) = 1 / (1 + e^(-z))
    # cuidado com overflow: use np.clip antes de calcular
    pass

def sigmoid_deriv(z):
    # SEU CÓDIGO AQUI
    # d/dz sigmoid(z) = sigmoid(z) * (1 - sigmoid(z))
    pass

def relu(z):
    # SEU CÓDIGO AQUI
    # ReLU(z) = max(0, z)
    pass

def relu_deriv(z):
    # SEU CÓDIGO AQUI
    # d/dz ReLU(z) = 1 se z > 0, 0 caso contrário
    pass


# --- VALIDAÇÃO ---
z = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

sig = sigmoid(z)
assert np.allclose(sig, [0.1192, 0.2689, 0.5, 0.7311, 0.8808], atol=1e-3), 'sigmoid incorreto'

sig_d = sigmoid_deriv(z)
assert np.allclose(sig_d, sig * (1 - sig), atol=1e-6), 'sigmoid_deriv incorreto'

r = relu(z)
assert np.allclose(r, [0, 0, 0, 1, 2]), 'relu incorreto'

r_d = relu_deriv(z)
assert np.allclose(r_d, [0, 0, 0, 1, 1]), 'relu_deriv incorreto'

print('✅ Exercício 4.1 correto!')

---
## Exercício 4.2 — Forward Pass

Arquitetura da rede: `Input(2) → Hidden(8, ReLU) → Output(1, Sigmoid)`

**Forward pass:**
- `Z1 = X @ W1 + b1`
- `A1 = relu(Z1)`
- `Z2 = A1 @ W2 + b2`
- `A2 = sigmoid(Z2)` ← saída (probabilidade)

In [ ]:
def inicializar_pesos(n_input, n_hidden, n_output):
    """
    Inicialização He para ReLU (melhor que inicialização aleatória simples).
    Retorna dicionário com W1, b1, W2, b2.
    """
    params = {
        'W1': np.random.randn(n_input, n_hidden) * np.sqrt(2.0 / n_input),
        'b1': np.zeros((1, n_hidden)),
        'W2': np.random.randn(n_hidden, n_output) * np.sqrt(2.0 / n_hidden),
        'b2': np.zeros((1, n_output))
    }
    return params


def forward(X, params):
    """
    Forward pass da rede.
    Retorna: (A2, cache) onde cache contém Z1, A1, Z2, A2 para o backprop.
    """
    # SEU CÓDIGO AQUI
    # Z1 = X @ W1 + b1
    # A1 = relu(Z1)
    # Z2 = A1 @ W2 + b2
    # A2 = sigmoid(Z2)
    # retorne A2 e cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2, 'X': X}
    pass


def binary_crossentropy(y_true, y_pred, eps=1e-9):
    # SEU CÓDIGO AQUI
    # BCE = -(1/n) * Σ [y*log(ŷ) + (1-y)*log(1-ŷ)]
    # use np.clip em y_pred para evitar log(0)
    pass


# --- VALIDAÇÃO ---
params = inicializar_pesos(n_input=2, n_hidden=8, n_output=1)
A2, cache = forward(X_tr, params)

assert A2.shape == (len(X_tr), 1), f'Shape esperado ({len(X_tr)}, 1), obtido {A2.shape}'
assert np.all(A2 >= 0) and np.all(A2 <= 1), 'Saída da sigmoid deve estar em [0, 1]'

loss = binary_crossentropy(y_tr, A2)
assert isinstance(loss, float) or loss.ndim == 0, 'Loss deve ser um escalar'
assert 0 < loss < 5, f'Loss inicial esperado entre 0 e 5, obtido {loss:.4f}'

print(f'Forward pass — Loss inicial: {loss:.4f}')
print('✅ Exercício 4.2 correto!')

---
## Exercício 4.3 — Backward Pass (Backpropagation)

O backprop calcula os gradientes da loss em relação a cada peso, usando a regra da cadeia.

**Gradientes (de trás para frente):**
- `dA2 = -(y/A2) + (1-y)/(1-A2)` ← derivada da BCE
- `dZ2 = dA2 * sigmoid_deriv(Z2)` ← regra da cadeia na sigmoid
- `dW2 = A1.T @ dZ2 / n`
- `db2 = mean(dZ2)`
- `dA1 = dZ2 @ W2.T`
- `dZ1 = dA1 * relu_deriv(Z1)` ← regra da cadeia na ReLU
- `dW1 = X.T @ dZ1 / n`
- `db1 = mean(dZ1)`

In [ ]:
def backward(y, params, cache):
    """
    Backward pass — calcula os gradientes via regra da cadeia.
    Retorna dicionário com dW1, db1, dW2, db2.
    """
    n = len(y)

    # SEU CÓDIGO AQUI
    # siga as fórmulas acima na ordem correta
    # retorne {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    pass


def atualizar_pesos(params, grads, lr):
    # SEU CÓDIGO AQUI
    # W = W - lr * dW para cada parâmetro
    # retorne os parâmetros atualizados
    pass


# --- VALIDAÇÃO: verificando que os gradientes reduzem o loss ---
params_test = inicializar_pesos(n_input=2, n_hidden=8, n_output=1)
A2_antes, cache_antes = forward(X_tr, params_test)
loss_antes = binary_crossentropy(y_tr, A2_antes)

grads = backward(y_tr, params_test, cache_antes)
params_test = atualizar_pesos(params_test, grads, lr=0.1)

A2_depois, _ = forward(X_tr, params_test)
loss_depois = binary_crossentropy(y_tr, A2_depois)

print(f'Loss antes:  {loss_antes:.6f}')
print(f'Loss depois: {loss_depois:.6f}')
assert loss_depois < loss_antes, 'O loss deve diminuir após uma atualização de pesos'
print('✅ Exercício 4.3 correto!')

---
## Exercício 4.4 — Training Loop Completo

In [ ]:
def treinar_rede(X_tr, y_tr, X_te, y_te, n_hidden=8, lr=0.1, epochs=1000):
    """
    Training loop completo.
    Retorna: (params, histórico_loss_treino, histórico_loss_val)
    """
    params = inicializar_pesos(X_tr.shape[1], n_hidden, 1)
    hist_treino, hist_val = [], []

    for epoch in range(epochs):
        # SEU CÓDIGO AQUI
        # 1. forward pass no treino
        # 2. calcule o loss de treino
        # 3. backward pass
        # 4. atualize os pesos
        # 5. calcule o loss de validação (apenas forward, sem atualizar pesos)
        # 6. armazene ambos os losses nos históricos
        pass

    return params, hist_treino, hist_val


# --- VALIDAÇÃO ---
params_final, hist_tr, hist_val = treinar_rede(
    X_tr, y_tr, X_te, y_te, n_hidden=16, lr=0.05, epochs=1000
)

# acurácia final
A2_final, _ = forward(X_te, params_final)
y_pred_final = (A2_final > 0.5).astype(int)
acc_final = accuracy_score(y_te, y_pred_final)

print(f'Acurácia final no teste: {acc_final:.4f}')
assert acc_final > 0.85, 'Acurácia esperada > 85% no dataset moons'
assert hist_tr[-1] < hist_tr[0], 'Loss de treino deve diminuir'

# curva de aprendizado
plt.figure(figsize=(9, 4))
plt.plot(hist_tr,  label='Treino',    color='steelblue')
plt.plot(hist_val, label='Validação', color='coral')
plt.title('Rede Neural do Zero — Curva de Aprendizado')
plt.xlabel('Época')
plt.ylabel('Binary Cross-Entropy')
plt.legend()
plt.tight_layout()
plt.show()
print('✅ Exercício 4.4 correto!')

**Reflexão:** o que acontece com o loss se você aumentar muito o learning rate? E se usar n_hidden muito pequeno? Teste e documente suas observações.

*(Escreva sua resposta aqui)*

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)